In [1]:
"""Lunar Lander : train a Lunar Lander, to land correctly on the moon."""
from pprint import pprint

import gymnasium as gym
import torch
from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login 
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from torch import nn, Tensor
from torch.distributions import Categorical

# How Gymnasium Works

- We create our environment using `gym.make()`
- We reset the environment to its initial state with `env.reset()`

Then, at each step
- Get an action using our model
- Using `env.step(action)`, we perform this action in the environment and get:
    - `observation` : The new state (st+1)
    - `reward` : The reward we get after executing the action
    - `terminated` : Indicates if the episode terminated (agent reach the terminal state)
    - `truncated` : Indicates a time limit or if an agent go out of bounds of the environment
    - `info` : A dictionary that provides additional information (depends on the environment)

In [ ]:
# First, we create our environment called LunarLander-v2
env = gym.make("LunarLander-v2")

# Then we reset this environment
observation, info = env.reset()

# see what the Environment looks like
# The observation is a vector of size 8
# each value contains different information about the lander
print("Observation Space Shape", env.observation_space.shape)

# Get a random observation
obs = env.observation_space.sample()
obs_fields = [
    "x",
    "y", 
    "x_velocity",
    "y_velocity",
    "angle",
    "angular_velocity",
    "left_leg_contact",
    "right_leg_contact"
]
print("Sample observation: ")
pprint(
    dict(zip(obs_fields, env.observation_space.sample(), strict=True)),
    sort_dicts=False
)

# The action space is a discrete space of 4 possible actions
print("\nAction Space Shape", env.action_space.n)
actions = [
    "Do nothing",
    "Fire left orientation engine",
    "Fire main engine",
    "Fire right orientation engine"
]
for i in range(20):
    # Get a random action
    action = env.action_space.sample()

    # Perform the action
    print(f"Step {i:>2} : {actions[action]}")
    observation, reward, terminated, truncated, info = env.step(action)

    # If the game is terminated (in our case we land, crashed) or truncated (timeout)
    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()

env.close()

## Reward function

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:
- Is increased/decreased the closer/further the lander is to the landing pad.
- Is increased/decreased the slower/faster the lander is moving.
- Is decreased the more the lander is tilted (angle not horizontal).
- Is increased by 10 points for each leg that is in contact with the ground.
- Is decreased by 0.03 points each frame a side engine is firing.
- Is decreased by 0.3 points each frame the main engine is firing.
- The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.

## Vectorized Environment
We create a vectorized environment (stack multiple independent environments into a single environment), this way, we’ll have more diverse experiences during the training.

In [2]:
n_envs = 10
vec_env = make_vec_env("LunarLander-v2", n_envs=n_envs)


observations = []
actions = []
rewards = []
is_active = []

# Restart the environment and get initial observation
observations.append(vec_env.reset())
while True:
    # Sample random actions
    acts_i = torch.randint(0, 4, (n_envs,)).numpy()
    actions.append(acts_i)

    # Perform the actions
    next_obs, rewards_i, done_i, info = vec_env.step(acts_i)
    rewards.append(rewards_i)
    is_active.append(~done_i)

    if done_i.any():
        break

    observations.append(next_obs)

observations = torch.as_tensor(observations, dtype=torch.float32)
actions = torch.as_tensor(actions, dtype=torch.int64)
rewards = torch.as_tensor(rewards, dtype=torch.float32)

vec_env.close()

In [14]:
print("Observations shape:", observations.shape)
print("Actions shape:", actions.shape)
print("Rewards shape:", rewards.shape)

Observations shape: torch.Size([62, 10, 8])
Actions shape: torch.Size([62, 10])
Rewards shape: torch.Size([62, 10])


# Simple Policy Network

In [ ]:
class PolicyNet(nn.Module):
    """Simple policy network."""

    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int) -> None:
        super().__init__()

        # Define the policy network
        self.pi = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, act_dim)
        )

    def forward(self, obs: Tensor) -> Tensor:
        """Get logits for actions."""
        return self.pi(obs)

    def get_policy(self, obs: Tensor) -> Categorical:
        """Compute distribution over actions given observations."""
        logits = self.forward(obs)
        return Categorical(logits=logits)

    def get_action(self, obs: Tensor) -> int:
        """Sample an action from the policy given an observation."""
        return self.get_policy(obs).sample().item()

model = PolicyNet(
    obs_dim=env.observation_space.shape[0],
    act_dim=env.action_space.n,
    hidden_dim=64
)

model

In [ ]:
def compute_loss(model: PolicyNet, obs: Tensor, act: Tensor, weights: Tensor) -> Tensor:
    """Compute the policy gradient loss from a batch of trajectories.

    - the (state, action, weights) triples are collected while acting
    according to the current policy.
    - the weight for a state-action pair is the return from the episode
    to which it belongs.

    Parameters
    ----------
    model : PolicyNet
        The policy network
    obs : Tensor[B, obs_dim]
        A batch of observations
    act : Tensor[B]
        A batch of actions
    weights : Tensor[B]
        Episode return for each state-action pair.

    Returns
    -------
    Tensor[1]
        The policy gradient loss
    """
    # Get action probability distribution given observation
    action_dist = model.get_policy(obs)

    # Compute log probability of the action
    logp: Tensor = action_dist.log_prob(value=act)

    # Compute the policy gradient loss
    return -(logp * weights).mean()

#  Proximal Policy Optimization